# Reproducing Hristova, Fasol et al. (2025) from dandiset 001888

**Paper.** Hristova K, Fasol MCM, McLaughlin N, Nawaz MS, Taskiran M, Buller-Peralta I, Harris AP,
Sutherland A, Bassi A, Ocampo-Garces A, Escudero J, Kind PC, Gonzalez-Sulser A (2025). Absence
seizures and sleep-wake abnormalities in a rat model of GRIN2B neurodevelopmental disorder.
*Epilepsia* 66:4996-5013. [doi:10.1111/epi.18606](https://doi.org/10.1111/epi.18606). This attempt
used the open-access full text (CC BY, PMC12779324) and its Supporting Information. After the first
verification round, the criterion of figs5b-transition-percent was corrected to the Supplementary
Methods' rule for SWDs initiated in NREM, and variants asked for by the verifier were added.

**Dandiset.** [DANDI:001888](https://dandiarchive.org/dandiset/001888/draft), version `draft`, as
modified on 2026-07-31 09:37:44 UTC. The paper cites no DANDI version (its data statement says raw
data are available on request), and the dandiset has no published version, so the draft is the
only version there is. A draft can change: `grin2b.check_version()` stops the notebook if the
draft's modification time or file count differs from the state read here.

**What the dandiset holds.** One NWB file per rat for 17 rats: the full 73 to 99 h wireless
recording (14 skull EEG channels, 2 neck EMG channels, 250.4 Hz), the two 24-h baseline windows
(BL1, BL2) in the epochs table, and per window the lab's automated outputs: sleep scores in 5-s
epochs, SWD events, an SWD flag per epoch, per-hour SWD totals and per-state spectra. The western
blots, development measures, visual scoring and pharmacology recordings of the paper are not in it.

**Analysis code.** The text of the paper and its Supporting Information was read from Europe PMC
(PMC12779324). The Supplementary Methods judge the state of an SWD in wake and REM on the epoch
before it, and SWD initiation in NREM and wake-NREM transitions on the six epochs before it.

**Animals.** The files give 14 Grin2b+/- (Het), 2 wild type (WT) and one rat without genotype or
sex (GRIN2B-132). The paper analysed 15 Het and 9 WT, so every genotype comparison here rests on
two WT rats. No data from outside the dandiset are used.

**Code read.** The lab's sleep scorer (`Gonzalez-Sulser-Team/AUTOMATIC-SLEEP-SCORER`, commit
95b25fb; R scorer, MATLAB channel selection and cepstral SWD detection) and SWD detector
(`Gonzalez-Sulser-Team/SWD-Automatic-Identification`, commit 3fd91f5; MATLAB cepstral detection).
The scorer shows how it puts SWD epochs into the sleep scores (the per-epoch SWD file value 6
becomes state 4) and that the stored spectra are its own (log10 multitaper amplitude referenced to
each frequency's 5th percentile), not the Welch spectra of the Methods. Here SWD epochs are the
epochs SWD events overlap, as the Methods have it, because the scorer's flags in the dandiset each
overlap an event, mostly as its last epoch, and cover about half of the epochs events overlap (data
check); the flags are a variant. Neither repository holds
the statistics or figure code.

| Claim | Source | Status |
|---|---|---|
| fig2b-swd-hourly: more SWDs per hour in Het, no hour effect | Fig. 2B | tested |
| fig2c-swd-number: more SWDs over 24 h | Fig. 2C | tested |
| fig2c-swd-mean-duration: longer SWDs | Fig. 2C | tested |
| fig2c-swd-total-duration: more SWD time | Fig. 2C | tested |
| disc-het-swd-minutes: Het 59.97 min, WT under 1 min | Discussion | tested |
| figs5a-swd-light-dark: no light-dark difference | Fig. S5A | tested |
| figs8-swd-sex: no sex effect on SWDs | Fig. S8B | tested |
| fig2d-swd-spectrum-peak: theta peak with harmonic | Fig. 2D | tested |
| fig2e-swd-delta-power: higher SWD delta power in Het | Fig. 2E | blocked (data missing: one WT with SWDs) |
| fig3d-rem-hourly: less REM per hour | Fig. 3D | tested |
| fig3g-rem-total, fig3g-rem-bouts, fig3g-rem-bout-duration | Fig. 3G | tested |
| fig3h-nrem-total, fig3h-nrem-bouts | Fig. 3H | tested |
| fig3i-wake-total, fig3i-wake-bouts | Fig. 3I | tested |
| fig4a-rem-spectrum, fig4b-nrem-beta, fig4c-wake-power | Fig. 4 | tested |
| fig5a-swd-state-distribution: 94.8% wake, 4.4% NREM, 0.5% REM | Fig. 5A | tested |
| fig5a-het-more-nrem-swds: larger NREM share in Het | Fig. 5A | blocked (data missing: one WT with SWDs) |
| fig5c-transition-swds: all Het have transitional SWDs | Fig. 5C | tested |
| figs5b-transition-percent: 24.2% transitional | Fig. S5B | tested |
| fig5f-swd-wake-correlation: R = -0.51 | Fig. 5F | tested |
| fig5def-no-other-correlations | Fig. 5D-F | tested |
| fig1c, fig1df: western blots | Fig. 1, S2 | blocked (data missing) |
| figs4-normal-development | Fig. S4 | blocked (data missing) |
| figs1-scorer-agreement | Table S1 | blocked (data missing) |
| fig6b, fig6c, fig6d, fig7c: pharmacology | Fig. 6, 7 | blocked (data missing) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from concurrent.futures import ThreadPoolExecutor
from scipy import signal, stats
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import grin2b

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
version = grin2b.check_version()
sessions = grin2b.load_all()
by_id = {s["subject_id"]: s for s in sessions}
print(f"dandiset {grin2b.DANDISET_ID} version {grin2b.DANDISET_VERSION} (modified {version['modified']}, "
      f"{version['asset_count']} assets): loaded {len(sessions)} files in {time.time() - started:.0f} s")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
rows = []
for s in sessions:
    rows.append({
        "subject": s["subject_id"], "sex": s["sex"], "genotype_stored": s["genotype_stored"],
        "genotype": s["genotype"], "age": s["age"], "strain": s["strain"],
        "session_start": s["session_start_time"][:19], "hours": round(s["eeg_shape"][0] / s["rate"] / 3600, 1),
        "eeg_ch": s["eeg_shape"][1], "emg_ch": s["emg_shape"][1], "rate_hz": s["rate"],
        "BL1_start_h": round(s["days"][0]["t0"] / 3600, 2), "BL2_start_h": round(s["days"][1]["t0"] / 3600, 2),
        "BL_lengths_h": sorted({round((d["t1"] - d["t0"]) / 3600, 4) for d in s["days"]}),
        "epoch_tags": list(s["epochs"].tags)})
subjects = pd.DataFrame(rows)
print(subjects.to_string(index=False))
print("\nSubject fields in the files:", sorted({tuple(s["subject_fields"]) for s in sessions}))
print("Files whose subject holds a genotype field:", sum("genotype" in s["subject_fields"] for s in sessions))
print("\nGenotype by sex:")
print(pd.crosstab(subjects.genotype.fillna("none"), subjects.sex).to_string())
print("Paper (Supplementary Methods): Het 8 male + 7 female = 15, WT 3 male + 6 female = 9 analysed;"
      " 4 Het and 9 WT more were recorded and excluded.")
print("\nRaw data: unit", sorted({s["unit"] for s in sessions}), "| conversion (V per code)",
      sorted({s["conversion"] for s in sessions}), "| offset", sorted({s["offset"] for s in sessions}),
      "| starting_time", sorted({s["starting_time"] for s in sessions}))
print("13 mV / 4096 codes =", 13e-3 / 4096, "V per code")
print("EEG channels:", sessions[0]["eeg_channels"], "| EMG channels:", sessions[0]["emg_channels"])
print("Same channel order in every file:", len({tuple(s["eeg_channels"]) for s in sessions}) == 1)
print("Storage chunk of the EEG (samples x channels):", sorted({s["eeg_chunks"] for s in sessions}))

Sixteen files give the genotype (`Het` or `Wt`) in the subject; GRIN2B-132 has no genotype and
sex `U`. With 14 Het (7 male, 7 female) and 2 WT (both female), the dandiset fits within the
paper's analysed groups (Het 8 male and 7 female, WT 3 male and 6 female) but holds 2 of its 9 WT;
it does not say which rats the paper analysed. All files are Long-Evans, aged P16W/P19W (a range, the same for all), recorded at
250.4 Hz with 14 EEG and 2 EMG channels, and each baseline window lasts exactly 24 h. The stated
conversion is 13 mV over 4096 codes, as the description says, with an offset of 0 (see the raw
values below).

In [ ]:
rows = []
for s in sessions:
    for d in s["days"]:
        st, fl = d["states"], d["flags"]
        ev = grin2b.event_epochs(d)
        onset_ep = (d["sz_start"] // grin2b.EPOCH_S).astype(int)
        z = s["seizure_totals_by_zt"]; z = z[z.baseline_window == d["bl"]].sort_values("zeitgeber_hour")
        hourly = np.bincount(np.floor(d["sz_start"] / 3600).astype(int), minlength=24)[:24]
        last = np.zeros(len(st), bool)
        last[np.minimum(np.ceil(d["sz_stop"] / grin2b.EPOCH_S).astype(int) - 1, len(st) - 1)] = True
        rows.append({
            "subject": s["subject_id"], "day": d["bl"][-1], "epochs": len(st),
            "grid_ok": bool(np.allclose(np.diff(d["epoch_start"]), 5) and abs(d["epoch_start"][0]) < 0.01
                            and np.allclose(d["flag_start"], d["epoch_start"])),
            "state_codes": "".join(map(str, sorted(set(st)))), "flag_values": "/".join(map(str, sorted(set(fl)))),
            "flag6_epochs": int((fl == 6).sum()), "swd_events": len(d["sz_start"]),
            "events_in_day": bool(np.all((d["sz_start"] >= 0) & (d["sz_stop"] <= 86400))),
            "dur_eq_stop_minus_start": bool(np.allclose(d["sz_stop"] - d["sz_start"], d["sz_dur"])) if len(d["sz_dur"]) else None,
            "dur_range_s": f"{d['sz_dur'].min():.1f}-{d['sz_dur'].max():.1f}" if len(d["sz_dur"]) else "",
            "onsets_in_flag6": round(float(np.mean(fl[onset_ep] == 6)), 3) if len(onset_ep) else np.nan,
            "flag6_with_event": round(float(np.mean(ev[fl == 6])), 3) if (fl == 6).any() else np.nan,
            "flag6_is_event_last_epoch": round(float(np.mean(last[fl == 6])), 3) if (fl == 6).any() else np.nan,
            "event_epochs_flagged": round(float(np.mean(fl[ev] == 6)), 3) if ev.any() else np.nan,
            "zt_swd_total_eq_events": int(z.spike_wave_discharge.sum()) == len(d["sz_start"]),
            "zt_swd_max_hour_diff": int(np.abs(z.spike_wave_discharge.to_numpy() - hourly).max()),
            "zt_n_events_row0_total": bool(z.number_of_seizure_events.iloc[0] == len(d["sz_start"])
                                          and (z.number_of_seizure_events.iloc[1:] == 0).all()),
            "zt_Day_row0_total": bool(z.Day.iloc[0] == len(d["sz_start"]) and (z.Day.iloc[1:] == 0).all()),
        })
tables = pd.DataFrame(rows)
print(tables.to_string(index=False))
print("\nTable descriptions:")
for k, v in sessions[0]["descriptions"].items():
    print(f"  {k}: {v}")

The scores are on a 5-s grid of 17,280 epochs per day starting at the baseline window, and every
SWD event falls inside its day with `duration` equal to stop minus start. Three descriptions do
not match the values:

- `sleep_states` is said to use code 4 for SWD epochs; no file holds code 4. The stored scores are
  the scorer's `dge_ok` output, before SWD epochs are set.
- `swd_epoch_counts` is said to hold the number of SWD events per epoch; it holds only 4 and 6.
  In the lab's scorer these are flags (6 = SWD epoch, 4 = none). Every flagged epoch overlaps an
  SWD event, and most flagged epochs are the last epoch an event overlaps, while only about a
  tenth of SWD onsets fall in a flagged epoch (columns `flag6_with_event`,
  `flag6_is_event_last_epoch`, `onsets_in_flag6`). The flags cover about half of the epochs SWD
  events overlap (`event_epochs_flagged`), so removing flagged epochs from the sleep states, as
  the scorer does, leaves the other SWD epochs in the states. The Methods exclude "SWD times";
  SWD epochs are therefore the epochs an SWD event overlaps here, and the flags are a variant.
- In `seizure_totals_by_zt` the per-hour SWD counts are in `spike_wave_discharge` (their total
  equals the number of events, and per hour they equal the events binned by start time to within
  one event at an hour boundary), while `number_of_seizure_events` and `Day` hold the day's total
  in the first row and 0 elsewhere. The description itself marks these columns as unconfirmed
  (TODO). Per-hour counts are taken from the events here.

GRIN2B-228 (Het) has 0 and 2 SWDs, and GRIN2B-238 (WT) has none; their files hold no
`seizure_events` table, or an empty one.

In [ ]:
rows = []
for s in sessions:
    for d in s["days"]:
        h = np.arange(len(d["states"])) * grin2b.EPOCH_S / 3600
        sleep = np.isin(d["states"], [1, 2])
        rows.append({"subject": s["subject_id"], "day": d["bl"][-1],
                     "sleep_frac_ZT0-12": sleep[h < 12].mean(), "sleep_frac_ZT12-24": sleep[h >= 12].mean()})
lightdark = pd.DataFrame(rows)
lightdark["light_minus_dark"] = lightdark["sleep_frac_ZT0-12"] - lightdark["sleep_frac_ZT12-24"]
print(lightdark.round(3).to_string(index=False))
print("\nDays with more sleep in ZT 0-12 than ZT 12-24:", int((lightdark.light_minus_dark > 0).sum()), "of", len(lightdark))

Rats sleep mostly in the light phase. Every day has more sleep in its first 12 h than in its
second, which is consistent with each baseline window starting at lights on (ZT 0), as the
description says. The files derive `session_start_time` from that assumption, so the absolute
clock time is not an independent check.

In [ ]:
rows = []
for s in sessions:
    sp = s["spectra"]
    for bl, g in sp.groupby("baseline_window"):
        f = g.frequency_hz.to_numpy()
        row = {"subject": s["subject_id"], "day": bl[-1], "rows": len(g), "f_min": f.min(), "f_max": f.max(),
               "f_step": np.unique(np.round(np.diff(f), 3)).tolist()}
        for c in ("Wake", "NREM", "REM", "SWD"):
            row[c] = f"{g[c].min():.3f}..{g[c].max():.3f}" if c in g else "absent"
        rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))
print("\nDescription:", sessions[0]["descriptions"]["sleep_state_power_spectra"])

The stored spectra run from 0 to 125.2 Hz in 0.2 Hz steps for each day and state, with no SWD
column for the two rats without SWDs in a day. Their values (0 to about 1.6) are not powers: the
lab's scorer writes the mean over a state's epochs of the log10 multitaper amplitude, referenced
per frequency to its 5th percentile over the day. The description (baseline-corrected PSD
normalized to the average over states, units to be confirmed) describes the Methods' Welch
pipeline, not these values. The spectral claims below use spectra computed from the raw EEG as
the Methods describe; the stored spectra are a variant.

In [ ]:
t_raw = time.time()


def raw_epochs(s):
    # Raw EEG (SWD channel) and EMG in one storage chunk at ZT 0, 4, 8, 12, 16, 20 of each day,
    # reduced at once to per-epoch spectra and EMG power, and 0.5-s SWD-band power.
    days = {d["bl"]: d for d in s["days"]}
    eps, eps_alt, swd_bins, codes = [], [], [], []
    for w in grin2b.raw_windows(s):
        d = days[w["bl"]]
        e = grin2b.window_epochs(w, s, d)
        e["bl"] = w["bl"]
        e["zt"] = w["zt"]
        eps.append(e)
        ea = grin2b.window_epochs(w, s, d, "eeg_alt")
        ea["bl"] = w["bl"]
        eps_alt.append(ea)
        t, p, ok = grin2b.swd_band_series(w, s)
        swd_bins.append({"bl": w["bl"], "t": t, "p": p, "ok": ok})
        x = w["eeg"]
        valid = x[(x >= 0) & (x <= grin2b.ADC_MAX)]
        codes.append((valid.min() if valid.size else np.nan, np.median(valid) if valid.size else np.nan,
                      valid.max() if valid.size else np.nan, (x == grin2b.DROPOUT).mean(),
                      ((x < 0) | ((x > grin2b.ADC_MAX) & (x != grin2b.DROPOUT))).mean()))
    e = pd.concat(eps, ignore_index=True)
    e["subject"] = s["subject_id"]
    return e, swd_bins, np.array(codes, float), pd.concat(eps_alt, ignore_index=True)


with ThreadPoolExecutor(6) as ex:
    raw = dict(zip(by_id, ex.map(raw_epochs, sessions)))
FREQ = grin2b.welch_freq()
rows = []
for sid, (e, _, c, _) in raw.items():
    rows.append({"subject": sid, "epochs": len(e), "epochs_with_bad_samples": round(e.dropout.mean(), 3),
                 "artefact_over_3000uV": round((e.artefact & ~e.dropout).mean(), 3),
                 "usable": int((~e.dropout & ~e.artefact).sum()),
                 "valid_code_min": np.nanmin(c[:, 0]), "valid_code_median": np.nanmedian(c[:, 1]),
                 "valid_code_max": np.nanmax(c[:, 2]), "samples_32767": round(c[:, 3].mean(), 4),
                 "samples_other_invalid": round(c[:, 4].mean(), 6)})
raw_summary = pd.DataFrame(rows)
print(f"read 12 raw windows per animal in {time.time() - t_raw:.0f} s")
print(raw_summary.to_string(index=False))

The raw EEG is stored as the amplifier's unsigned 12-bit codes: valid samples sit around a median
near 2600, not around 0, so with the stated offset of 0 every channel reads about +8 mV of DC.
Signal loss is written as 32767 on every channel at once, a value the description does not
mention, and some files also hold other values outside 0 to 4095 (negative, or between 4096 and
32766), which a 12-bit amplifier cannot produce. Epochs touching any of these are dropped here;
the fraction varies widely between rats (column `epochs_with_bad_samples`; 0.73 for GRIN2B-238 in
the windows read). The DC level is removed before any spectrum.

In [ ]:
# Clock check: do the sleep scores and SWD events line up with the raw signals they were scored from?
emg_shifts = np.arange(-60, 61)
swd_shifts = np.arange(-10, 10.5, 0.5)
rows, emg_curves, swd_curves = [], {}, {}
for s in sessions:
    sid = s["subject_id"]
    e, bins = raw[sid][:2]
    days = {d["bl"]: d for d in s["days"]}
    ok = ~e.dropout
    curve = []
    for k in emg_shifts:
        lab = np.concatenate([days[bl]["states"][np.clip(g.epoch.to_numpy() - k, 0, grin2b.EPOCHS_PER_DAY - 1)]
                              for bl, g in e[ok].groupby("bl")])
        v = np.concatenate([g.emg_logpow.to_numpy() for bl, g in e[ok].groupby("bl")])
        curve.append(v[lab == 0].mean() - v[lab == 1].mean())
    emg_curves[sid] = np.array(curve)
    row = {"subject": sid, "emg_wake_minus_nrem_at_0": round(curve[60], 3),
           "emg_best_shift_epochs": int(emg_shifts[np.argmax(curve)]), "emg_at_+-60": round((curve[0] + curve[-1]) / 2, 3)}
    n_in = sum(grin2b.in_events(b["t"], days[b["bl"]]["sz_start"], days[b["bl"]]["sz_stop"]).sum() for b in bins)
    if n_in >= 20:
        c2 = []
        for sh in swd_shifts:
            inside, outside = [], []
            for b in bins:
                m = grin2b.in_events(b["t"], days[b["bl"]]["sz_start"], days[b["bl"]]["sz_stop"], sh)
                inside.append(b["p"][m & b["ok"]]); outside.append(b["p"][~m & b["ok"]])
            c2.append(np.concatenate(inside).mean() - np.concatenate(outside).mean())
        swd_curves[sid] = np.array(c2)
        row.update({"swd_bins_inside": int(n_in), "swd_contrast_at_0": round(c2[20], 3),
                    "swd_best_shift_s": float(swd_shifts[np.argmax(c2)]), "swd_contrast_at_+-10s": round((c2[0] + c2[-1]) / 2, 3)})
    else:
        row.update({"swd_bins_inside": int(n_in)})
    # Per-epoch 5-10 Hz power, against the epochs SWD events overlap and the epochs the scorer flags,
    # each shifted by k epochs.
    ep_p = []
    for b in bins:
        k = np.floor(b["t"] / grin2b.EPOCH_S).astype(int)
        df_ = pd.DataFrame({"epoch": k[b["ok"]], "p": b["p"][b["ok"]]}).groupby("epoch").p.mean()
        ep_p.append((b["bl"], df_))
    if n_in >= 20:
        for name, lab_fn in (("events", grin2b.event_epochs), ("flags", lambda d: d["flags"] == grin2b.SWD_FLAG)):
            best = []
            for k in range(-3, 4):
                ins, outs = [], []
                for bl, pe in ep_p:
                    lab = lab_fn(days[bl])
                    idx = np.clip(pe.index.to_numpy() - k, 0, grin2b.EPOCHS_PER_DAY - 1)
                    ins.append(pe.to_numpy()[lab[idx]]); outs.append(pe.to_numpy()[~lab[idx]])
                ins = np.concatenate(ins)
                best.append(ins.mean() - np.concatenate(outs).mean() if ins.size else np.nan)
            row[f"{name}_epoch_best_shift"] = int(np.arange(-3, 4)[np.nanargmax(best)])
            row[f"{name}_epoch_contrast_at_0"] = round(best[3], 3)
    rows.append(row)
clock = pd.DataFrame(rows)
print("EMG: log10 60-90 Hz power (right minus left EMG), mean in wake minus mean in NREM epochs, with the scores shifted by k epochs.")
print("SWD: log10 5-10 Hz EEG power in 0.5-s bins, mean inside minus outside SWD events, with the events shifted by s seconds.")
print("events/flags_epoch: the same power per 5-s epoch, in epochs overlapped by an event or flagged 6 by the scorer, minus the others,"
      " with the labels shifted by k epochs (best of -3..3).")
print(clock.to_string(index=False))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for sid, c in emg_curves.items():
    ax[0].plot(emg_shifts * 5, c, lw=0.8)
for sid, c in swd_curves.items():
    ax[1].plot(swd_shifts, c, lw=0.8)
ax[0].set(xlabel="shift of the sleep scores (s)", ylabel="EMG wake - NREM (log10)")
ax[1].set(xlabel="shift of the SWD events (s)", ylabel="5-10 Hz inside - outside (log10)")
plt.tight_layout(); plt.show()
print("plotted ranges: EMG", np.round([min(c.min() for c in emg_curves.values()), max(c.max() for c in emg_curves.values())], 3),
      "| SWD", np.round([min(c.min() for c in swd_curves.values()), max(c.max() for c in swd_curves.values())], 3))

For 15 of the 17 rats the EMG separates wake from NREM best with the scores where they are; for
GRIN2B-129 the best shift is one epoch and for GRIN2B-139 it is 20 epochs, where the curve is flat
(column `emg_best_shift_epochs`; bouts are long, so the contrast changes little over a few epochs,
and it is lower at 60 epochs in every rat). For every rat with enough SWD time in
the raw windows, the 5-10 Hz power inside SWD events is highest with the events where they are
and falls off within seconds. The scores and events share the clock of the raw recording in the
windows read. Rats with too few SWD bins in the windows (`swd_bins_inside` below 20) are not
checked for SWD timing.

At the 5-s scale, the epochs overlapped by SWD events contrast best with the labels unshifted in
every rat checked, and the flagged epochs at 0 or -1 epochs (`flags_epoch_best_shift`), with a
similar contrast at 0; the flags are on the same clock, consistent with their marking the end of
an SWD.

In [ ]:
ids = list(by_id)
agree = []
for i, a in enumerate(ids):
    for b in ids[i + 1:]:
        for k in range(2):
            sa, sb = by_id[a]["days"][k]["states"], by_id[b]["days"][k]["states"]
            ta, tb = by_id[a]["days"][k]["sz_start"], by_id[b]["days"][k]["sz_start"]
            agree.append({"pair": f"{a} {b}", "day": k + 1, "score_agreement": (sa == sb).mean(),
                          "identical_swd_times": len(np.intersect1d(np.round(ta, 2), np.round(tb, 2)))})
agree = pd.DataFrame(agree)
print("Highest agreement of sleep scores between two rats on the same day index:")
print(agree.sort_values("score_agreement", ascending=False).head(5).round(3).to_string(index=False))
n_ev = {sid: [len(d["sz_start"]) for d in by_id[sid]["days"]] for sid in ids}
agree["expected_by_chance"] = [n_ev[p.split()[0]][d - 1] * n_ev[p.split()[1]][d - 1] / (86400 / 0.2)
                               for p, d in zip(agree.pair, agree.day)]
print("Shared SWD start times (on the 0.2-s detection grid): largest", int(agree.identical_swd_times.max()),
      "| largest expected by chance", round(agree.expected_by_chance.max(), 1),
      "| pairs above 3 times chance and above 5:", int(((agree.identical_swd_times > 3 * agree.expected_by_chance) & (agree.identical_swd_times > 5)).sum()))
same_date = subjects.groupby("session_start").subject.apply(list)
rows = []
for date, group in same_date.items():
    for i, a in enumerate(group):
        for b in group[i + 1:]:
            ea, eb = raw[a][0], raw[b][0]
            m = ea.merge(eb, on=["bl", "epoch"], suffixes=("_a", "_b"))
            m = m[~m.dropout_a & ~m.dropout_b]
            rows.append({"pair": f"{a} {b}", "epochs": len(m),
                         "corr_emg_logpow": np.corrcoef(m.emg_logpow_a, m.emg_logpow_b)[0, 1] if len(m) > 10 else np.nan})
print("\nRats recorded together (same session start): correlation of per-epoch EMG power")
print(pd.DataFrame(rows).round(3).to_string(index=False))

No two files share their scores beyond what two rats in the same light cycle do by chance, SWD
start times coincide between files only as often as chance predicts on the 0.2-s detection grid,
and the EMG power of rats recorded together is correlated (they share a room and light cycle) but
far from identical. No recording
appears to be deposited twice. GRIN2B-227 and GRIN2B-229 have the same number of samples, which
follows from being recorded together.

Per animal, the measures below are the means over the two baseline days. Every genotype test is
also shown without GRIN2B-228 (Het, 1 SWD per day), without GRIN2B-238 (WT, no SWDs) and without
both, as the dandiset does not say whether they are among the paper's analysed rats. With
GRIN2B-238 left out one WT remains: a rank-sum test then cannot reach p < 0.05 and a Welch test is
not defined. Two more variants concern SWD epochs: removing only the SWD time from the states
(event start to stop) where the main choice removes whole epochs, and forming bouts with SWD epochs
dropped, so that an SWD does not split a bout.

In [ ]:
def per_animal(mode="flag", bridge=False, exact=False):
    # bridge: SWD epochs do not split bouts; exact: state minutes less the SWD time itself.
    rows = []
    for s in sessions:
        days = []
        for d in s["days"]:
            st = grin2b.scored_states(d, mode)
            m = grin2b.state_measures(st, bridge)
            if exact:
                m.update(grin2b.exact_minutes(d))
            m.update({k: v for k, v in grin2b.swd_measures(d).items() if k != "swd_hourly"})
            m["swd_epochs"] = int((st == 4).sum())
            days.append(m)
        r = pd.DataFrame(days).mean().to_dict()
        r["subject"] = s["subject_id"]
        r["sex"] = s["sex"]
        r["genotype"] = s["genotype"]
        rows.append(r)
    return pd.DataFrame(rows).set_index("subject")


animals = {m: per_animal(m) for m in ("events", "flag", "stored")}
animals["bridged"] = per_animal("events", bridge=True)
animals["exact"] = per_animal("events", exact=True)
A = animals["events"]
cols = ["genotype", "sex", "swd_n", "swd_mean_s", "swd_total_min", "swd_epochs", "wake_min", "nrem_min", "rem_min",
        "wake_bouts", "nrem_bouts", "rem_bouts", "wake_bout_s", "nrem_bout_s", "rem_bout_s"]
print(A[cols].sort_values(["genotype", "sex"]).round(2).to_string())
print("\nCounts:", A.genotype.fillna("none").value_counts().to_dict())
het = A[A.genotype == "Het"]
print(f"Het mean SWD time {het.swd_total_min.mean():.2f} min per day; SWD epochs removed from the states "
      f"{het.swd_epochs.mean() * grin2b.EPOCH_S / 60:.2f} min per day; wake minutes with whole SWD epochs removed "
      f"{het.wake_min.mean():.1f}, with only the SWD time removed {animals['exact'][animals['exact'].genotype == 'Het'].wake_min.mean():.1f}")

# Sensitivity: GRIN2B-228 (Het, 1 SWD per day) and GRIN2B-238 (WT, no SWDs) may not be among the
# paper's analysed rats (the Supplementary Methods excluded 4 Het and 9 WT of one founder line).
DROPS = {"without 228": ["GRIN2B-228"], "without 238": ["GRIN2B-238"],
         "without 228 and 238": ["GRIN2B-228", "GRIN2B-238"]}


def variants(measure, table=None):
    # The groups under the main choice and its variants: GRIN2B-132 as WT or as Het.
    t = A if table is None else table
    out = {"main (132 left out)": t[t.genotype.notna()].assign(g=t.genotype)}
    for as_ in ("WT", "Het"):
        out[f"132 as {as_}"] = t.assign(g=t.genotype.fillna(as_))
    for name, ids in DROPS.items():
        out[name] = out["main (132 left out)"][~out["main (132 left out)"].index.isin(ids)]
    return out


def compare(measure, test, direction, extra_tables=()):
    # The paper's two-group test under the main choice and each variant; direction "higher"/"lower" for Het.
    rows = []
    tabs = [("", A)] + list(extra_tables)
    for tname, t in tabs:
        for vname, d in variants(measure, t).items():
            if tname and vname != "main (132 left out)":
                continue
            d = d[d[measure].notna()]
            wt, het = d[d.g == "WT"][measure], d[d.g == "Het"][measure]
            r = grin2b.two_group(wt, het, test)
            diff = het.mean() - wt.mean()
            rows.append({"variant": (tname + " " + vname).strip(), "n_WT": len(wt), "n_Het": len(het),
                         "WT_mean": wt.mean(), "Het_mean": het.mean(), "Het_minus_WT": diff,
                         "test": r["test"], "stat": r["stat"], "df": r["df"], "p": r["p"],
                         "paper_direction": (diff > 0) if direction == "higher" else (diff < 0)})
    return pd.DataFrame(rows)

## fig2b-swd-hourly

> Grin2b+/- rats have more SWDs than WT across the 24-h day; hour of day has no effect and does not
> interact with genotype (linear mixed model, genotype F = 55.36, p < .00001; hour p = .16;
> interaction p = .49).

In [ ]:
def hourly_table(fn):
    rows = []
    for s in sessions:
        v = np.mean([fn(d) for d in s["days"]], axis=0)
        for h in range(len(v)):
            rows.append({"subject": s["subject_id"], "hour": h, "value": v[h], "genotype": s["genotype"]})
    return pd.DataFrame(rows)


def hourly_tests(H, within="hour"):
    out = []
    for vname, g in (("main (132 left out)", H[H.genotype.notna()]),
                     ("132 as WT", H.assign(genotype=H.genotype.fillna("WT"))),
                     ("132 as Het", H.assign(genotype=H.genotype.fillna("Het"))),
                     *((name, H[H.genotype.notna() & ~H.subject.isin(ids)]) for name, ids in DROPS.items())):
        a = grin2b.split_plot_anova(g, "value", "subject", "genotype", within)
        a.insert(0, "variant", vname)
        out.append(a)
    return pd.concat(out, ignore_index=True)


def mixed_check(H, within="hour"):
    # The paper's model: a linear mixed model with a random intercept per animal (REML), Wald tests
    # of each term with sum contrasts, on the main animals; compared with the split-plot F tests.
    g = H[H.genotype.notna()]
    fit = sm.MixedLM.from_formula(f"value ~ C(genotype, Sum) * C({within}, Sum)", data=g, groups=g.subject).fit(reml=True)
    w = fit.wald_test_terms(scalar=True).table
    w = w[w.index != "Intercept"]
    return pd.DataFrame({"term": w.index, "chi2_over_df": w.statistic.to_numpy() / w.df_constraint.to_numpy(),
                         "df": w.df_constraint.to_numpy(), "p": w.pvalue.to_numpy(), "converged": fit.converged})


H_swd = hourly_table(lambda d: grin2b.swd_measures(d)["swd_hourly"])
print("Mean SWDs per hour by genotype (main):")
print(H_swd[H_swd.genotype.notna()].groupby(["genotype", "hour"]).value.mean().unstack().round(1).to_string())
print(hourly_tests(H_swd).round(5).to_string(index=False))
print("Mixed model (MixedLM, random intercept), main animals:")
print(mixed_check(H_swd).round(5).to_string(index=False))

**Result: reproduced.** Het have more SWDs per hour than WT (genotype F = 11.45, p = 0.0045),
hour has no effect at p < 0.05 (p = 0.069), and it does not interact with genotype (p = 0.9998).
Het means per hour range from 19.2 to 37.8, WT means from 0 to 8.2. With GRIN2B-132 as WT or as
Het the genotype effect holds (p = 0.0011 and 0.013), and hour reaches p = 0.041 in both. Without
GRIN2B-228, without GRIN2B-238 or without both, genotype p = 0.0001, 0.048 and 0.0036, hour p = 0.070
to 0.074 and interaction p >= 0.9997. A random-intercept mixed model (MixedLM) gives the same
genotype and interaction statistics (genotype 11.45, Wald p = 0.0007; interaction p = 0.9998); its
hour term (p = 0.9998) differs from the split-plot hour term, as it weighs the two groups equally.
The comparison rests on two WT rats.

## fig2c-swd-number

> Grin2b+/- rats have more SWDs over 24 h than WT (Wilcoxon rank-sum, W = 4.5, p = .0002).

In [ ]:
print(A[["genotype", "swd_n"]].sort_values(["genotype", "swd_n"]).round(3).to_string())
print("Animals without a value (no SWDs):", A[A["swd_n"].isna()].index.tolist())
print(compare("swd_n", "wilcoxon", "higher").round(5).to_string(index=False))

**Result: reproduced.** Het have 690.6 SWDs per day on average against 56.3 in WT (W = 1, p = 0.047).
GRIN2B-238 has none; GRIN2B-240 (112.5 per day) ranks above one Het, GRIN2B-228 (1 per day). With
GRIN2B-132 as WT p = 0.020, as Het p = 0.062. Without GRIN2B-228 p = 0.034 (W = 0); with GRIN2B-238
left out one WT remains and p = 0.20 (0.14 without both).

## fig2c-swd-mean-duration

> SWDs are longer on average in Grin2b+/- rats than in WT (two-sample t test, df = 22, T = -5.26, p = .00003).

In [ ]:
print(A[["genotype", "swd_mean_s"]].sort_values(["genotype", "swd_mean_s"]).round(3).to_string())
print("Animals without a value (no SWDs):", A[A["swd_mean_s"].isna()].index.tolist())
print(compare("swd_mean_s", "t", "higher").round(5).to_string(index=False))

**Result: partially reproduced.** Het SWDs last 4.54 s on average against 1.47 s in the only WT rat
with SWDs (GRIN2B-240), the paper's direction, but the t test with one WT gives p = 0.16. With
GRIN2B-132 (1.70 s) as WT p = 0.062, and without GRIN2B-228 p = 0.090. A t test against one WT
value says little either way.

## fig2c-swd-total-duration

> The total time in SWDs over 24 h is higher in Grin2b+/- rats than in WT (Wilcoxon rank-sum, W = 5.5, p = .0002).

In [ ]:
print(A[["genotype", "swd_total_min"]].sort_values(["genotype", "swd_total_min"]).round(3).to_string())
print("Animals without a value (no SWDs):", A[A["swd_total_min"].isna()].index.tolist())
print(compare("swd_total_min", "wilcoxon", "higher").round(5).to_string(index=False))

**Result: reproduced.** Het spend 57.77 min per day in SWDs against 1.37 min in WT (W = 1, p = 0.033).
With GRIN2B-132 as WT p = 0.012, as Het p = 0.029. Without GRIN2B-228 p = 0.019; with GRIN2B-238
left out one WT remains and p = 0.27 (0.14 without both).

## disc-het-swd-minutes

> Over 24 h, Grin2b+/- rats spend on average 59.97 min in SWDs, while WT littermates average less
> than a minute.

In [ ]:
for vname, d in variants("swd_total_min").items():
    g = d.groupby("g").swd_total_min
    print(f"{vname}: Het mean {g.mean()['Het']:.2f} min (n {g.size()['Het']}, SEM {g.sem()['Het']:.2f}), "
          f"WT mean {g.mean()['WT']:.2f} min (n {g.size()['WT']}; values {sorted(d[d.g == 'WT'].swd_total_min.round(2))})")
print("Expected Het 59.97 +- 6.0 min; WT below 1 min.")
print("Het mean without GRIN2B-228:", round(A[(A.genotype == "Het") & (A.index != "GRIN2B-228")].swd_total_min.mean(), 2))
print("SWD time from the epochs SWD events overlap (5 s each), Het mean:", round(A[A.genotype == "Het"].swd_epochs.mean() * 5 / 60, 2), "min")
F = animals["flag"]
print("SWD time from the epochs the scorer flags (5 s each), Het mean:", round(F[F.genotype == "Het"].swd_epochs.mean() * 5 / 60, 2), "min")

**Result: partially reproduced.** The Het mean is 57.77 min per day (SEM 9.04, 14 rats), within the
tolerance of 59.97 +- 6.0 min. The WT mean is 1.37 min (0 and 2.75 min), above the paper's less
than a minute. Without GRIN2B-228 the Het mean is 62.21 min, also within tolerance. Without GRIN2B-238 the WT mean is
2.75 min, so the WT part fails with either WT set. Counting each epoch that an SWD event overlaps
as 5 s of SWD gives 110.57 min, and counting each epoch the scorer flags gives 60.2 min, so the
paper's number fits both the event durations and the flags.

## figs5a-swd-light-dark

> The number of SWDs does not differ between the light and dark phases, and Grin2b+/- rats have
> more SWDs than WT in both (linear mixed model, genotype p < .00001, phase p = .28, interaction
> p = .53).

In [ ]:
H_ld = hourly_table(lambda d: np.array([grin2b.swd_measures(d)["swd_light"], grin2b.swd_measures(d)["swd_dark"]]))
H_ld["phase"] = np.where(H_ld.hour == 0, "light", "dark")
print(H_ld[H_ld.genotype.notna()].pivot_table(index="subject", columns="phase", values="value").join(A.genotype).round(1).to_string())
print(H_ld[H_ld.genotype.notna()].groupby(["genotype", "phase"]).value.mean().round(1).to_string())
print(hourly_tests(H_ld.drop(columns="hour"), within="phase").round(5).to_string(index=False))
het = H_ld[H_ld.genotype == "Het"].pivot_table(index="subject", columns="phase", values="value")
print("Het light minus dark: median", round((het.light - het.dark).median(), 1), "| Wilcoxon signed-rank p",
      round(stats.wilcoxon(het.light, het.dark).pvalue, 4))

**Result: reproduced.** Phase has no effect (F = 1.19, p = 0.29) and genotype has (F = 11.45, p =
0.0045), with Het higher in both phases (316.7 SWDs in the light and 373.9 in the dark, against
12.5 and 43.8 in WT) and no interaction (p = 0.86). Within Het, light minus dark has a median of
-79.8 SWDs (signed-rank p = 0.19). The variants with GRIN2B-132 give phase p = 0.27. Without GRIN2B-228, GRIN2B-238 or both, phase p =
0.30 and genotype p = 0.0001, 0.048 and 0.0036.

## figs8-swd-sex

> Sex does not affect the number, mean duration or total duration of SWDs (two-way ANOVA sex x
> genotype, sex p = .74, .99 and .99).

In [ ]:
d = A[A.genotype.notna() & A.sex.isin(["M", "F"])].copy()
print(pd.crosstab(d.genotype, d.sex).to_string())
rows = []
for measure in ("swd_n", "swd_mean_s", "swd_total_min"):
    dd = d[d[measure].notna()]
    fit = ols(f"{measure} ~ C(genotype) + C(sex)", data=dd).fit()
    a = sm.stats.anova_lm(fit, typ=2)
    het = dd[dd.genotype == "Het"]
    t = stats.ttest_ind(het[het.sex == "M"][measure], het[het.sex == "F"][measure])
    rows.append({"measure": measure, "n": len(dd), "sex_F": a.loc["C(sex)", "F"], "sex_p": a.loc["C(sex)", "PR(>F)"],
                 "Het_male_mean": het[het.sex == "M"][measure].mean(), "Het_female_mean": het[het.sex == "F"][measure].mean(),
                 "Het_only_t_p": t.pvalue})
print(pd.DataFrame(rows).round(4).to_string(index=False))
for name, ids in DROPS.items():
    dd0 = d[~d.index.isin(ids)]
    ps = []
    for measure in ("swd_n", "swd_mean_s", "swd_total_min"):
        dd = dd0[dd0[measure].notna()]
        ps.append(round(sm.stats.anova_lm(ols(f"{measure} ~ C(genotype) + C(sex)", data=dd).fit(), typ=2).loc["C(sex)", "PR(>F)"], 3))
    print(f"{name}: sex p for number, mean duration, total duration = {ps}")

**Result: reproduced.** The sex term has p = 0.29, 0.94 and 0.78 for the number, mean duration and
total duration of SWDs. Within Het, females have 763.3 SWDs per day against 617.9 in males (t test
p = 0.31), 4.59 against 4.50 s (p = 0.94) and 60.3 against 55.2 min (p = 0.79). The ANOVA has no
interaction term, as there are no WT males. Without GRIN2B-228, GRIN2B-238 or both the sex terms
have p from 0.31 to 0.94.

## Spectra from the raw EEG

The Methods' spectra, on the epochs of the raw windows: usable epochs only, each epoch's Welch
spectrum divided by the animal's mean spectrum over its usable wake, NREM and REM epochs, then
averaged per state (SWD epochs are those SWD events overlap). Absolute spectra are kept for the SWD
peak. Variants: SWD epochs from the scorer's flags; the spectral-slope rejection of the Methods
(slope of log10 power against log10 frequency over 1 to 48 Hz below -5, or power at 1 Hz above
500 mV^2/Hz, a reading of the stated units chosen here); and, per animal, whichever S1 channel
(right or left) has fewer epochs above the artefact threshold, as the Methods chose the cleaner
channel by eye for sleep scoring.

In [ ]:
def slope_bad(e):
    return ((e.slope < grin2b.SLOPE_MIN) | (10 ** e.offset > grin2b.OFFSET_MAX)).to_numpy()


def artefact_rate(e):
    ok = ~e.dropout
    return (e.artefact[ok]).mean()


# Per animal, the S1 channel with fewer artefact epochs (ties keep S1_Tr_R).
cleaner = {sid: ("S1_Tr_L" if artefact_rate(raw[sid][3]) < artefact_rate(raw[sid][0]) else "S1_Tr_R") for sid in raw}


def state_spectra(mode="events", slope_reject=False, channel="main"):
    out = {}
    for s in sessions:
        sid = s["subject_id"]
        e = raw[sid][3] if channel == "cleaner" and cleaner[sid] == "S1_Tr_L" else raw[sid][0]
        days = {d["bl"]: d for d in s["days"]}
        sc = {bl: grin2b.scored_states(d, mode) for bl, d in days.items()}
        st = np.array([sc[bl][k] for bl, k in zip(e.bl, e.epoch)])
        use = (~e.dropout & ~e.artefact).to_numpy()
        if slope_reject:
            use = use & ~slope_bad(e)
        P = np.vstack(e.psd.to_numpy())
        base = P[use & np.isin(st, [0, 1, 2])].mean(0)
        r = {"n": {}}
        for c, name in grin2b.STATES.items():
            m = use & (st == c)
            r["n"][name] = int(m.sum())
            r[name] = (P[m] / base).mean(0) if m.any() else None
            r[name + "_abs"] = P[m].mean(0) if m.any() else None
        out[sid] = r
    return out


spectra = state_spectra("events")
spectra_flag = state_spectra("flag")
spectra_slope = state_spectra("events", slope_reject=True)
spectra_clean = state_spectra("events", channel="cleaner")
print("Usable epochs per state in the raw windows:")
print(pd.DataFrame({sid: r["n"] for sid, r in spectra.items()}).T.join(A.genotype).to_string())

# Coverage: usable epochs of each state in the windows, as a fraction of that state's epochs over both days.
rows = []
for s in sessions:
    sid = s["subject_id"]
    e = raw[sid][0]
    tot = {c: sum(int((grin2b.scored_states(d, "events") == c).sum()) for d in s["days"]) for c in (0, 1, 2, 4)}
    use = ~e.dropout & ~e.artefact
    row = {"subject": sid, "genotype": s["genotype"], "window_epochs_frac_of_2_days": round(len(e) / (2 * grin2b.EPOCHS_PER_DAY), 3)}
    for c, name in ((0, "Wake"), (1, "NREM"), (2, "REM"), (4, "SWD")):
        row[f"{name}_covered"] = round(spectra[sid]["n"][name] / tot[c], 3) if tot[c] else np.nan
    row["slope_rule_removes"] = int((use & slope_bad(e)).sum())
    row["slope_min"] = round(e.slope[use].min(), 2)
    row["offset_1Hz_max_mV2_per_Hz"] = float(f"{(10 ** e.offset[use]).max():.3g}")
    row["artefact_rate_R"] = round(artefact_rate(raw[sid][0]), 3)
    row["artefact_rate_L"] = round(artefact_rate(raw[sid][3]), 3)
    row["cleaner_channel"] = cleaner[sid]
    rows.append(row)
coverage = pd.DataFrame(rows)
print("\nCoverage of each state by the usable epochs of the windows, the slope rule and the cleaner S1 channel:")
print(coverage.to_string(index=False))
print("Median coverage:", coverage[[c for c in coverage if c.endswith("_covered")]].median().round(3).to_dict(),
      "| epochs the slope rule removes, all animals:", int(coverage.slope_rule_removes.sum()),
      "| animals where S1_Tr_L is cleaner:", coverage[coverage.cleaner_channel == "S1_Tr_L"].subject.tolist())


def band_table(state, bands=grin2b.BANDS, spec=None):
    spec = spectra if spec is None else spec
    rows = []
    for sid, r in spec.items():
        if r[state] is None:
            continue
        for b, v in grin2b.band_power(FREQ, r[state], bands).items():
            rows.append({"subject": sid, "band": b, "value": v, "genotype": by_id[sid]["genotype"]})
    return pd.DataFrame(rows)


def stored_band_table(state, bands=grin2b.BANDS):
    rows = []
    for s in sessions:
        sp = s["spectra"]
        if state not in sp:
            continue
        g = sp.groupby("frequency_hz")[state].mean()
        f = g.index.to_numpy()
        for b, (lo, hi) in bands.items():
            rows.append({"subject": s["subject_id"], "band": b, "value": g[(f >= lo) & (f < hi)].mean(),
                         "genotype": s["genotype"]})
    return pd.DataFrame(rows)


def band_anova(T, name):
    rows = []
    for vname, d in (("main (132 left out)", T[T.genotype.notna()]),
                     ("132 as WT", T.assign(genotype=T.genotype.fillna("WT"))),
                     ("132 as Het", T.assign(genotype=T.genotype.fillna("Het"))),
                     *((name, T[T.genotype.notna() & ~T.subject.isin(ids)]) for name, ids in DROPS.items())):
        d = d.dropna(subset=["value"])
        fit = ols("value ~ C(genotype) * C(band)", data=d).fit()
        a = sm.stats.anova_lm(fit, typ=2)
        tk = pairwise_tukeyhsd(d.value, d.genotype + ":" + d.band)
        tk = pd.DataFrame(tk.summary().data[1:], columns=tk.summary().data[0])
        row = {"spectra": name, "variant": vname, "n_WT": d[d.genotype == "WT"].subject.nunique(),
               "n_Het": d[d.genotype == "Het"].subject.nunique(),
               "genotype_F": a.loc["C(genotype)", "F"], "genotype_p": a.loc["C(genotype)", "PR(>F)"],
               "interaction_F": a.loc["C(genotype):C(band)", "F"], "interaction_p": a.loc["C(genotype):C(band)", "PR(>F)"]}
        means = d.groupby(["genotype", "band"]).value.mean()
        row["mean_Het-WT"] = np.mean([means.get(("Het", b), np.nan) - means.get(("WT", b), np.nan) for b in grin2b.BANDS])
        for b in grin2b.BANDS:
            pair = tk[((tk.group1 == f"Het:{b}") & (tk.group2 == f"WT:{b}")) | ((tk.group1 == f"WT:{b}") & (tk.group2 == f"Het:{b}"))]
            row[f"{b}_Het-WT"] = means.get(("Het", b), np.nan) - means.get(("WT", b), np.nan)
            row[f"{b}_tukey_p"] = float(pair["p-adj"].iloc[0]) if len(pair) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def spectral_claim(state):
    res = [band_anova(band_table(state), "Methods, raw"),
           band_anova(band_table(state, grin2b.BANDS_ALT), "Methods, raw, contiguous bands"),
           band_anova(band_table(state, spec=spectra_flag), "Methods, raw, SWD epochs from scorer flags"),
           band_anova(band_table(state, spec=spectra_slope), "Methods, raw, slope rejection"),
           band_anova(band_table(state, spec=spectra_clean), "Methods, raw, cleaner S1 channel"),
           band_anova(stored_band_table(state), "stored (scorer)")]
    return pd.concat(res, ignore_index=True)

The usable epochs of the windows cover a median of 5.0% of each rat's wake epochs over the two days,
9.5% of NREM, 8.2% of REM and 6.8% of SWD epochs; the lowest are GRIN2B-129's REM (1.1%, 4 epochs)
and GRIN2B-238's wake (1.5%). The slope rule, read as above, removes no epoch: the steepest slope is
-3.76 and the largest power at 1 Hz is 5.34 mV^2/Hz. S1_Tr_L has fewer artefact epochs than S1_Tr_R
in 7 rats, among them both WT; the artefact rate is at most 3.7% on either channel. The band ANOVAs
below treat the five bands of an animal as separate observations, as the paper's two-way ANOVA
does, so their p-values are smaller than a test on animals would give.

## fig2d-swd-spectrum-peak

> The EEG spectrum during SWDs has a clear peak in the theta range with a harmonic, in both
> genotypes.

In [ ]:
rows = []
band = (FREQ >= 3) & (FREQ <= 30)
for sid, r in spectra.items():
    n = r["n"]["SWD"]
    row = {"subject": sid, "genotype": by_id[sid]["genotype"], "swd_epochs": n}
    if n >= 5:
        lp = np.log10(r["SWD_abs"][band])
        f = FREQ[band]
        pk, prop = signal.find_peaks(lp, prominence=0.1)
        peaks = f[pk]
        # any peak at 5-10 Hz with another peak within 1 Hz of twice its frequency
        pairs = [(p, q) for p in peaks if 5 <= p <= 10 for q in peaks if abs(q - 2 * p) <= 1]
        row["peaks_hz"] = ", ".join(f"{p:.1f}" for p in peaks)
        row["f0_hz"] = pairs[0][0] if pairs else np.nan
        row["harmonic_hz"] = pairs[0][1] if pairs else np.nan
        row["meets"] = bool(pairs)
        stored = by_id[sid]["spectra"]
        if "SWD" in stored:
            g = stored.groupby("frequency_hz").SWD.mean()
            gb = g[(g.index >= 3) & (g.index <= 30)]
            spk, _ = signal.find_peaks(gb.to_numpy(), prominence=0.02)
            row["stored_peaks_hz"] = ", ".join(f"{p:.1f}" for p in gb.index[spk][:4])
    rows.append(row)
peak = pd.DataFrame(rows)
print(peak.to_string(index=False))
ok = peak[peak.swd_epochs >= 5]
print("\nAnimals with >= 5 usable SWD epochs:", len(ok), "| meeting the criterion:", int(ok.meets.sum()),
      "| by genotype:", ok.groupby(ok.genotype.fillna("none")).meets.agg(["sum", "size"]).to_dict("index"))
fig, ax = plt.subplots(figsize=(6, 3.5))
for sid, r in spectra.items():
    if r["n"]["SWD"] >= 5:
        ax.semilogy(FREQ[band], r["SWD_abs"][band], lw=0.8, color="C3" if by_id[sid]["genotype"] == "WT" else "C0")
ax.set(xlabel="Hz", ylabel="SWD power (uV^2/Hz)", title="SWD epochs; WT red, Het and unlabelled blue")
plt.tight_layout(); plt.show()
vals = np.concatenate([r["SWD_abs"][band] for r in spectra.values() if r["n"]["SWD"] >= 5])
print("plotted range (uV^2/Hz):", np.round([vals.min(), vals.max()], 2))

**Result: reproduced.** All 15 rats with at least 5 usable SWD epochs in the raw windows, 13 Het, the
WT GRIN2B-240 and GRIN2B-132, have a spectral peak at 7.0 to 8.0 Hz and a second peak at 14.0 to
16.0 Hz, twice its frequency; 12 of them also have a third at 21 to 24 Hz. The stored spectra
show peaks near the same frequencies (7.4 to 8.2 and 14.4 to 16.2 Hz). GRIN2B-240 has 18 usable SWD epochs, the fewest.

## fig3d-rem-hourly

> Grin2b+/- rats have less REM sleep per hour than WT across the day, and the reduction is not
> specific to certain hours (linear mixed model, genotype F = 9.98, p = .0045; hour x genotype p = .53).

In [ ]:
H_rem = hourly_table(lambda d: grin2b.hourly_minutes(grin2b.scored_states(d, "events"), 2))
print("Mean REM minutes per hour by genotype (main):")
print(H_rem[H_rem.genotype.notna()].groupby(["genotype", "hour"]).value.mean().unstack().round(2).to_string())
print(hourly_tests(H_rem).round(5).to_string(index=False))
print("Mixed model (MixedLM, random intercept), main animals:")
print(mixed_check(H_rem).round(5).to_string(index=False))
for mode in ("flag", "stored"):
    H_v = hourly_table(lambda d: grin2b.hourly_minutes(grin2b.scored_states(d, mode), 2))
    print(f"Variant, SWD epochs mode {mode}:")
    print(hourly_tests(H_v).round(5).head(3).to_string(index=False))

**Result: reproduced.** Het have less REM per hour than WT (genotype F = 4.69, p = 0.048), and the
reduction does not depend on the hour (interaction p = 0.096). The effect is at the threshold:
with GRIN2B-132 as WT p = 0.19, as Het p = 0.038 (interaction p = 0.071), and with the scorer's
flags or the stored scores p = 0.049 and 0.050. Without GRIN2B-228 genotype p = 0.060; without
GRIN2B-238 or both genotype p = 0.019 and 0.025, but the interaction then has p = 0.0001 and
0.00005. The outcome holds with all 16 rats and with GRIN2B-132 counted as Het, and not without
GRIN2B-228 or GRIN2B-238. The mixed model gives genotype 4.69 (Wald p = 0.030) and interaction
p = 0.086.

## fig3g-rem-total

> Grin2b+/- rats spend less total time in REM sleep over 24 h than WT (two-sample t test, df = 22, T = 3.16, p = .0045).

In [ ]:
print(A[["genotype", "rem_min"]].sort_values(["genotype", "rem_min"]).round(2).to_string())
print(compare("rem_min", "t", "lower",
              [("scorer flags", animals["flag"]), ("stored scores", animals["stored"]), ("SWD time only", animals["exact"])]).round(5).to_string(index=False))

**Result: reproduced.** Het have 44.8 min of REM per day against 69.7 in WT (t = 2.16, df = 14, p =
0.048). The result is at the threshold: with GRIN2B-132 as WT p = 0.19, as Het p = 0.038, and the
two SWD-epoch variants give p = 0.049 and 0.050. Both WT (54.3 and 85.2 min) are above the Het
median, and 5 Het rats have more REM than GRIN2B-238. Without GRIN2B-228 p = 0.060; without
GRIN2B-238 p = 0.019, without both 0.025. Removing only the SWD time gives p = 0.049.

## fig3g-rem-bouts

> Grin2b+/- rats have fewer REM bouts than WT (two-sample t test, df = 22, T = 3.94, p = .00069).

In [ ]:
print(A[["genotype", "rem_bouts"]].sort_values(["genotype", "rem_bouts"]).round(2).to_string())
print(compare("rem_bouts", "t", "lower",
              [("scorer flags", animals["flag"]), ("stored scores", animals["stored"]), ("bridged bouts", animals["bridged"])]).round(5).to_string(index=False))

**Result: reproduced.** Het have 37.9 REM bouts per day against 74.0 in WT (t = 4.12, df = 14, p =
0.0010); both WT (60.5 and 87.5) have more than every Het (at most 55.5). With GRIN2B-132 as WT p =
0.068, as Het p = 0.0008; the SWD-epoch variants give p = 0.0008 and 0.0007. Without GRIN2B-228, GRIN2B-238 or both p = 0.0017,
0.0007 and 0.0011; with bouts not split by SWD epochs p = 0.0007.

## fig3g-rem-bout-duration

> REM bouts are longer on average in Grin2b+/- rats than in WT (Wilcoxon rank-sum, W = 33, p = .043).

In [ ]:
print(A[["genotype", "rem_bout_s"]].sort_values(["genotype", "rem_bout_s"]).round(2).to_string())
print(compare("rem_bout_s", "wilcoxon", "higher",
              [("scorer flags", animals["flag"]), ("stored scores", animals["stored"]), ("bridged bouts", animals["bridged"])]).round(5).to_string(index=False))

**Result: partially reproduced.** Het REM bouts last 71.9 s on average against 56.8 s in WT, the
paper's direction, with W = 2 and p = 0.067. With GRIN2B-132 (95.5 s) as WT the difference is 2.2 s
(p = 0.59). Without GRIN2B-228 p = 0.076; with one WT p = 0.27 and 0.29. With bouts not split by SWD
epochs p = 0.067.

## fig3h-nrem-total

> Total NREM sleep over 24 h does not differ between genotypes (two-sample t test, df = 22, T = -.09, p = .93).

In [ ]:
print(A[["genotype", "nrem_min"]].sort_values(["genotype", "nrem_min"]).round(2).to_string())
print(compare("nrem_min", "t", "none",
              [("scorer flags", animals["flag"]), ("stored scores", animals["stored"]), ("SWD time only", animals["exact"])]).round(5).to_string(index=False))

**Result: reproduced.** The t test finds no difference in NREM time (Het 450.5 against WT 419.3 min
per day, difference 31.2 min, t = -1.29, p = 0.22). With GRIN2B-132 as WT it does (48.4 min, p =
0.035); as Het p = 0.37. Without GRIN2B-228 p = 0.21; without GRIN2B-238 the Het have 17.6 min less
(p = 0.54), and 15.9 min less without both (p = 0.59). Removing only the SWD time gives p = 0.19.

## fig3i-wake-total

> Grin2b+/- rats spend less total time awake over 24 h than WT (Welch t test, df = 10.25, T = 2.26, p = .047).

In [ ]:
print(A[["genotype", "wake_min"]].sort_values(["genotype", "wake_min"]).round(2).to_string())
print(compare("wake_min", "welch", "lower",
              [("scorer flags", animals["flag"]), ("stored scores", animals["stored"]), ("SWD time only", animals["exact"])]).round(5).to_string(index=False))

**Result: partially reproduced.** Het are awake 834.1 min per day against 944.9 in WT (difference
-110.8 min), the paper's direction, but the Welch test gives p = 0.35 with two WT. The difference
depends on how SWDs are taken out of wake: it is -65.5 min (p = 0.50) when only the SWD time is
removed, -68.1 min (p = 0.49) with the scorer's flags and -10.8 min (p = 0.89) with the stored
scores. Without GRIN2B-228 it is -121.2 min (p = 0.33); with one WT the Welch test is not defined.

## fig3h-nrem-bouts

> Grin2b+/- rats have fewer NREM bouts, which are longer, than WT (Wilcoxon rank-sum, bouts W = 29.5, p = .025; mean bout duration W = 26, p = .015).

In [ ]:
print(A[["genotype", "nrem_bouts", "nrem_bout_s"]].sort_values("genotype").round(2).to_string())
print("\nnrem_bouts:")
print(compare("nrem_bouts", "wilcoxon", "lower", [("scorer flags", animals["flag"]), ("stored scores", animals["stored"]), ("bridged bouts", animals["bridged"])]).round(5).to_string(index=False))
print("\nnrem_bout_s:")
print(compare("nrem_bout_s", "wilcoxon", "higher", [("scorer flags", animals["flag"]), ("stored scores", animals["stored"]), ("bridged bouts", animals["bridged"])]).round(5).to_string(index=False))

**Result: partially reproduced.** Het NREM bouts are longer than in WT (127.1 against 92.3 s, W = 0,
p = 0.017; both WT below every Het). Het have fewer NREM bouts (218.5 against 273.0 per day), the
paper's direction, but p = 0.10. With the scorer's flags, the stored scores or bouts not split by SWD
epochs both parts have p < 0.05 (bouts p = 0.032, duration p = 0.017). Without GRIN2B-228 bouts p =
0.11 and duration p = 0.019; with one WT neither test can reach p < 0.05.

## fig3i-wake-bouts

> Grin2b+/- rats have more wake bouts of shorter duration than WT (Wilcoxon rank-sum, W = 18, p = .0035; Welch t test on mean bout duration, df = 8.91, T = 2.97, p = .016).

In [ ]:
print(A[["genotype", "wake_bouts", "wake_bout_s"]].sort_values("genotype").round(2).to_string())
print("\nwake_bouts:")
print(compare("wake_bouts", "wilcoxon", "higher", [("scorer flags", animals["flag"]), ("stored scores", animals["stored"]), ("bridged bouts", animals["bridged"])]).round(5).to_string(index=False))
print("\nwake_bout_s:")
print(compare("wake_bout_s", "welch", "lower", [("scorer flags", animals["flag"]), ("stored scores", animals["stored"]), ("bridged bouts", animals["bridged"])]).round(5).to_string(index=False))

**Result: partially reproduced.** Het have more wake bouts (559.4 against 294.5 per day, W = 2, p =
0.067) and shorter ones (100.6 against 206.2 s, Welch p = 0.31), both in the paper's direction
without p < 0.05. With the stored scores, in which SWD epochs count as wake, both reverse (Het 190.8
against 241.0 bouts, p = 0.017; 304.2 against 239.2 s, p = 0.19), and with bouts not split by SWD
epochs they reverse too (Het 186.3 against 241.0 bouts, p = 0.017; 275.4 against 237.7 s, p = 0.38),
so the result depends on SWD epochs ending wake bouts. Without GRIN2B-228 the bout count has p =
0.019 (W = 0) and the bout duration p = 0.29.

## fig4a-rem-spectrum

> EEG spectral power during REM sleep does not differ between genotypes (two-way ANOVA genotype x band, genotype p = .14, interaction p = .69).

In [ ]:
T = band_table("REM")
print(T[T.genotype.notna()].pivot_table(index=["genotype", "subject"], columns="band", values="value")[list(grin2b.BANDS)].round(3).to_string())
print("Het minus WT, mean over bands:", round(T[T.genotype == "Het"].value.mean() - T[T.genotype == "WT"].value.mean(), 4))
print(spectral_claim("REM").round(4).to_string(index=False))

**Result: reproduced.** For normalized REM band power neither genotype (F = 3.30, p = 0.073) nor the
interaction (p = 0.998) has p < 0.05; Het are lower by 0.19 on average across bands. The genotype
effect reaches p = 0.034 with contiguous bands and p < 0.0001 with SWD epochs from the scorer's
flags, and has p = 0.17 with the stored spectra. The slope rule removes no epoch and leaves the result
unchanged; with the cleaner S1 channel genotype p = 0.067 and interaction p = 0.9993. Without
GRIN2B-228 p = 0.094, without GRIN2B-238 or both p = 0.19 and 0.22.

## fig4b-nrem-beta

> During NREM sleep Grin2b+/- rats have higher beta power than WT (two-way ANOVA genotype x band, interaction p = .038; Tukey beta p = .0008, other bands p >= .82).

In [ ]:
T = band_table("NREM")
print(T[T.genotype.notna()].pivot_table(index=["genotype", "subject"], columns="band", values="value")[list(grin2b.BANDS)].round(3).to_string())
print("Het minus WT, mean over bands:", round(T[T.genotype == "Het"].value.mean() - T[T.genotype == "WT"].value.mean(), 4))
print(spectral_claim("NREM").round(4).to_string(index=False))

**Result: not reproduced.** NREM beta power is lower in Het than in WT (difference -0.078 in
normalized power, Tukey p = 1.0), and the genotype by band interaction has p = 0.90. In every
variant with both WT, Het beta is lower, by 0.0035 to 0.30. With GRIN2B-238 left out, Het beta is
higher in 7 of the 12 variants, by 0.0032 to 0.147, with Tukey p >= 0.99 and interaction p >= 0.92.

## fig4c-wake-power

> During wake, spectral power is reduced in Grin2b+/- rats overall, without a difference in single bands (two-way ANOVA genotype x band, genotype p = .01, interaction p = .86).

In [ ]:
T = band_table("Wake")
print(T[T.genotype.notna()].pivot_table(index=["genotype", "subject"], columns="band", values="value")[list(grin2b.BANDS)].round(3).to_string())
print("Het minus WT, mean over bands:", round(T[T.genotype == "Het"].value.mean() - T[T.genotype == "WT"].value.mean(), 4))
print(spectral_claim("Wake").round(4).to_string(index=False))

**Result: not reproduced.** Normalized wake power is higher in Het than in WT on average across
bands (+0.081; genotype F = 0.58, p = 0.45). Averaged over the bands, Het are higher in every
variant on the raw EEG with both WT (+0.0065 to +0.168, column `mean_Het-WT`). Het are lower on
average with the stored spectra (-0.027, genotype p = 0.59; p = 0.0072 only when GRIN2B-132 is
counted as WT), and with the cleaner S1 channel without GRIN2B-238 (-0.0072, p = 0.93; -0.016
without both, p = 0.86).


## fig5a-swd-state-distribution

> Over both genotypes, SWDs occur mostly in wake (94.8%), then NREM (4.4%), and rarely in REM sleep (0.5%).

In [ ]:
def onsets(rule):
    rows = []
    for s in sessions:
        for d in s["days"]:
            o = grin2b.swd_onsets(d, rule)
            o["subject"] = s["subject_id"]
            rows.append(o)
    return pd.concat(rows, ignore_index=True)


on = {rule: onsets(rule) for rule in ("prev", "any6")}
rows = []
for rule, o in on.items():
    per = pd.crosstab(o.subject, o.state, normalize="index") * 100
    per = per.reindex(columns=[0, 1, 2], fill_value=0)
    pooled = o.state.value_counts(normalize=True).reindex([0, 1, 2], fill_value=0) * 100
    rows.append({"reading": f"({'a' if rule == 'prev' else 'c'}) mean over animals, NREM rule {rule}", "animals": len(per),
                 "wake": per[0].mean(), "NREM": per[1].mean(), "REM": per[2].mean()})
    rows.append({"reading": f"({'b' if rule == 'prev' else 'd'}) pooled SWDs, NREM rule {rule}", "animals": len(per),
                 "wake": pooled[0], "NREM": pooled[1], "REM": pooled[2]})
dist = pd.DataFrame(rows).sort_values("reading")
dist["all_within_tolerance"] = ((dist.wake - 94.8).abs() <= 9.48) & ((dist.NREM - 4.4).abs() <= 0.44) & ((dist.REM - 0.5).abs() <= 0.05)
print(dist.round(3).to_string(index=False))
print("Expected 94.8 +- 9.48, 4.4 +- 0.44, 0.5 +- 0.05. SWDs:", len(on["prev"]))
print("Readings (c) and (d) use the Supplementary Methods' rule: wake and REM from the epoch before the SWD,"
      " NREM when any of the six epochs before it is NREM.")
per_c = (pd.crosstab(on["any6"].subject, on["any6"].state, normalize="index") * 100).reindex(columns=[0, 1, 2], fill_value=0)
for name, ids in DROPS.items():
    for lab, pp in (("(a)", None), ("(c)", per_c)):
        q = (pd.crosstab(on["prev"].subject, on["prev"].state, normalize="index") * 100).reindex(columns=[0, 1, 2], fill_value=0) if pp is None else pp
        q = q[~q.index.isin(ids)]
        print(f"{name}, reading {lab}: wake {q[0].mean():.2f}, NREM {q[1].mean():.2f}, REM {q[2].mean():.2f} ({len(q)} animals)")
per = (pd.crosstab(on["prev"].subject, on["prev"].state, normalize="index") * 100).reindex(columns=[0, 1, 2], fill_value=0)
print(per.join(A.genotype).join(A.swd_n).round(2).to_string())

**Result: partially reproduced.** Under reading (a), 94.2% of SWDs follow wake, 5.5% NREM and 0.27%
REM: wake is within tolerance and the order holds, but NREM is above 4.4 +- 0.44 and REM below 0.5
+- 0.05. Reading (c), the Supplementary Methods' rule (wake and REM from the epoch before, NREM
from the six epochs before), gives 91.2%, 8.6% and 0.19%, with the same pattern. No reading puts
all three within tolerance (pooled, reading (b): 92.9%, 6.8% and 0.32%). Without GRIN2B-228 the
values move by at most 0.6 percentage points.

## fig5c-transition-swds

> All Grin2b+/- rats have SWDs that start at wake-NREM transitions, against 3 of 9 WT rats
> (Fisher's exact test, p = .00062).

In [ ]:
o = on["prev"]
tr = o.groupby("subject").agg(swds=("state", "size"), nrem_swds=("state", lambda x: int((x == 1).sum())),
                              transitional=("transition", "sum"))
tr = A[["genotype", "swd_n"]].join(tr).fillna({"swds": 0, "nrem_swds": 0, "transitional": 0})
tr["has_transitional"] = tr.transitional > 0
print(tr.sort_values("genotype").to_string())
for vname, d in variants("swd_n", tr).items():
    tab = pd.crosstab(d.g, d.has_transitional).reindex(index=["Het", "WT"], columns=[True, False], fill_value=0)
    p = stats.fisher_exact(tab.to_numpy())[1]
    print(f"{vname}: Het {tab.loc['Het', True]}/{tab.loc['Het'].sum()}, WT {tab.loc['WT', True]}/{tab.loc['WT'].sum()}, Fisher p = {p:.4f}")

**Result: partially reproduced.** 13 of 14 Het rats have at least one SWD at a wake-NREM transition,
against 0 of 2 WT (Fisher p = 0.025). The exception is GRIN2B-228, which has 2 SWDs in two days,
none after NREM. GRIN2B-132 has none. Without
GRIN2B-228 every Het counts (13 of 13, Fisher p = 0.0095), which would meet the criterion; with
GRIN2B-238 left out one WT remains and p = 0.13 (0.071 without both). The outcome is judged on all
16 rats.

## figs5b-transition-percent

> In Grin2b+/- rats 24.2% of SWDs occur during the wake-NREM transition, similar to the 22.39% of
> the WT rats that have such SWDs (two-sample t test, p = .74).

In [ ]:
# Reading (i) follows the Supplementary Methods: an SWD is initiated in NREM when any of the six epochs
# before it is NREM (rule any6), and the share of these that are at a wake-NREM transition is taken.
# Every transitional SWD has an NREM epoch among its six epochs before, so it is in this denominator.
# The first draft of the criterion took NREM from the epoch before (rule prev); that is shown as (i-prev),
# with all transitional SWDs in the numerator, as first run, and with only those whose epoch before is NREM.
o6 = on["any6"]
tr["nrem_swds_any6"] = o6.groupby("subject").state.apply(lambda x: int((x == 1).sum())).reindex(tr.index).fillna(0)
tr["trans_prev_nrem"] = o.groupby("subject").apply(lambda g: int((g.transition & (g.state == 1)).sum())).reindex(tr.index).fillna(0)
tr["pct_i_methods"] = 100 * tr.transitional / tr.nrem_swds_any6.where(tr.nrem_swds_any6 > 0)
tr["pct_i_prev_as_run"] = 100 * tr.transitional / tr.nrem_swds.where(tr.nrem_swds > 0)
tr["pct_i_prev_restricted"] = 100 * tr.trans_prev_nrem / tr.nrem_swds.where(tr.nrem_swds > 0)
tr["pct_ii_all"] = 100 * tr.transitional / tr.swds.where(tr.swds > 0)
cols = ["pct_i_methods", "pct_i_prev_as_run", "pct_i_prev_restricted", "pct_ii_all"]
print(tr[["genotype", "swds", "nrem_swds_any6", "nrem_swds", "transitional", "trans_prev_nrem"] + cols].sort_values("genotype").round(2).to_string())
names = {"pct_i_methods": "(i) of NREM-initiated SWDs, Methods rule (six epochs)",
         "pct_i_prev_as_run": "(i-prev) epoch-before NREM rule, as first run",
         "pct_i_prev_restricted": "(i-prev) epoch-before NREM rule, numerator restricted to these SWDs",
         "pct_ii_all": "(ii) of all SWDs"}
for col, name in names.items():
    het = tr[(tr.genotype == "Het")][col].dropna()
    wt = tr[(tr.genotype == "WT") & tr.has_transitional][col].dropna()
    line = f"{name}: Het mean {het.mean():.2f}% (n {len(het)}, SEM {het.sem():.2f}); WT with transitional SWDs: {wt.round(2).tolist()}"
    for dname, ids in DROPS.items():
        line += f" | {dname}: {het[~het.index.isin(ids)].mean():.2f}"
    print(line)
print("Expected Het 24.2 +- 2.42%")


**Result: reproduced.** The Supplementary Methods judge SWD initiation in NREM on the six epochs
before the SWD. Under that rule, 22.93% (SEM 1.97, 13 Het) of the Het SWDs initiated in NREM are at a
wake-NREM transition (reading (i)), within 24.2 +- 2.42%. The first draft of the criterion took
NREM from the epoch before the SWD, which gives 36.3%; it was corrected after the first
verification round. Reading (ii), the share of all SWDs, gives 2.2%. Leaving out GRIN2B-228, which
has no SWD initiated in NREM, does not change reading (i). No WT rat has transitional SWDs, so the
WT value and the t test cannot be computed.


## fig5f-swd-wake-correlation

> In Grin2b+/- rats, total SWD duration is negatively correlated with total time awake (Pearson,
> R = -.51, p = .048).

In [ ]:
rows = []
for tname, t in (("main", A), ("scorer flags", animals["flag"]), ("stored scores", animals["stored"]),
                 ("SWD time only", animals["exact"])):
    for vname, sel in (("Het", t.genotype == "Het"), ("Het and 132", t.genotype.fillna("Het") == "Het"),
                       ("Het without 228", (t.genotype == "Het") & (t.index != "GRIN2B-228"))):
        d = t[sel]
        r, p = stats.pearsonr(d.swd_total_min, d.wake_min)
        rows.append({"scores": tname, "animals": vname, "n": len(d), "R": r, "p": p})
print(A[A.genotype == "Het"][["swd_total_min", "wake_min"]].round(2).sort_values("swd_total_min").to_string())
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: reproduced.** Over the 14 Het, total SWD time and wake time correlate at R = -0.81 (p =
0.0005), stronger than the paper's -0.51 and inside the tolerance. With the scorer's flags R = -0.75
(p = 0.0019). With only the SWD time removed from wake R = -0.74 (p = 0.0024). With the stored scores, in which SWD
epochs count as wake, R = -0.06 (p = 0.83). The correlation therefore comes from SWD time being
taken out of wake: in Het, 110.6 min of SWD epochs per day against 57.8 min of SWD time. Without
GRIN2B-228 R = -0.73 (p = 0.0043).

## fig5def-no-other-correlations

> In Grin2b+/- rats, total SWD duration is not correlated with total REM or NREM time, the number of
> REM, NREM or wake bouts, or the mean REM, NREM or wake bout duration (Pearson, all p >= .066).

In [ ]:
measures = ["rem_min", "nrem_min", "rem_bouts", "nrem_bouts", "wake_bouts", "rem_bout_s", "nrem_bout_s", "wake_bout_s"]
paper_r = {"rem_min": -.015, "nrem_min": -.13, "rem_bouts": -.19, "nrem_bouts": .49, "wake_bouts": .17,
           "rem_bout_s": .17, "nrem_bout_s": -.37, "wake_bout_s": -.25}
rows = []
for tname, t in (("main", A), ("scorer flags", animals["flag"]), ("stored scores", animals["stored"]),
                 ("bridged bouts", animals["bridged"]), ("SWD time only", animals["exact"]),
                 ("main without 228", A[A.index != "GRIN2B-228"])):
    d = t[t.genotype == "Het"]
    for m in measures:
        r, p = stats.pearsonr(d.swd_total_min, d[m])
        rows.append({"scores": tname, "measure": m, "n": len(d), "R": r, "p": p, "paper_R": paper_r[m]})
corr = pd.DataFrame(rows)
print(corr.round(4).to_string(index=False))
print("\nTests with p < 0.05:", corr.groupby("scores").p.apply(lambda p: int((p < 0.05).sum())).to_dict())

**Result: partially reproduced.** Six of the eight measures are not correlated with total SWD time
(p from 0.16 to 0.96), but the number of wake bouts (R = 0.54, p = 0.046) and the mean wake bout
duration (R = -0.58, p = 0.028) are: two of eight. With the scorer's flags the same two correlate
(R = 0.82 and -0.72); with the stored scores, with bouts not split by SWD epochs, or without
GRIN2B-228, none of the eight does (wake bouts R = 0.13, -0.09 and 0.29).

## Blocked claims

Two genotype comparisons need at least two animals with SWDs in each genotype, and only one WT
rat (GRIN2B-240) has SWDs; the paper's other seven WT are not in the dandiset. The cell below also
runs both analyses with GRIN2B-132 counted as WT, as an illustration only, since its genotype is
unknown:

In [ ]:
with_swd = A[A.swd_n > 0]
print("Animals with SWDs by genotype:", with_swd.genotype.fillna("none").value_counts().to_dict())
print("Animals with >= 5 usable SWD epochs in the raw windows:",
      {g: int(n) for g, n in pd.Series({sid: r["n"]["SWD"] for sid, r in spectra.items()}).ge(5).groupby(
          pd.Series({sid: by_id[sid]["genotype"] or "none" for sid in spectra})).sum().items()})

# Not a test of either claim: GRIN2B-132 has no genotype. With 132 counted as WT, two WT have SWDs.
print("\nfig2e-swd-delta-power, normalized SWD band power, animals with >= 5 usable SWD epochs:")
n_swd = {sid: r["n"]["SWD"] for sid, r in spectra.items()}
Tswd = band_table("SWD")
Tswd = Tswd[Tswd.subject.map(n_swd) >= 5]
bw = band_anova(Tswd, "Methods, raw, SWD")
print(bw[bw.variant.isin(["main (132 left out)", "132 as WT"])].round(4).to_string(index=False))
print("\nfig5a-het-more-nrem-swds, per-animal SWD state percentages (reading (a)), 132 as WT:")
pa = (pd.crosstab(on["prev"].subject, on["prev"].state, normalize="index") * 100).reindex(columns=[0, 1, 2], fill_value=0)
pa.columns = ["Wake", "NREM", "REM"]
pa = pa.join(A.genotype.fillna("WT").rename("g"))
print(pa.groupby("g").agg(["mean", "size"]).round(2).to_string())
long = pa.reset_index().melt(id_vars=["subject", "g"], var_name="state", value_name="pct")
a = sm.stats.anova_lm(ols("pct ~ C(g) * C(state)", data=long).fit(), typ=2)
print(a.round(4).to_string())
tk = pairwise_tukeyhsd(long.pct, long.g + ":" + long.state)
tk = pd.DataFrame(tk.summary().data[1:], columns=tk.summary().data[0])
for st_ in ("Wake", "NREM", "REM"):
    r = tk[tk.group1.str.endswith(":" + st_) & tk.group2.str.endswith(":" + st_)]
    print(f"Tukey {st_} Het-WT: p = {float(r['p-adj'].iloc[0]):.4f}")
print(f"\ntotal run time {time.time() - started:.0f} s")

With GRIN2B-132 as WT, SWD delta power is lower in Het than in the two WT (-0.34, Tukey p = 1.0),
against the paper's higher delta in Het; the genotype by band interaction has p = 0.027, from beta
(+6.8, Tukey p = 0.023). The share of SWDs after NREM is 6.30% in Het against 0.23% in the two WT
(interaction p = 0.013, Tukey NREM p = 0.29). Neither is a test of the claims.

## Summary

| Claim | Outcome |
|---|---|
| fig2b-swd-hourly | reproduced |
| fig2c-swd-number | reproduced |
| fig2c-swd-mean-duration | partially reproduced |
| fig2c-swd-total-duration | reproduced |
| disc-het-swd-minutes | partially reproduced |
| figs5a-swd-light-dark | reproduced |
| figs8-swd-sex | reproduced |
| fig2d-swd-spectrum-peak | reproduced |
| fig2e-swd-delta-power | blocked (data missing: one WT with SWDs) |
| fig3d-rem-hourly | reproduced |
| fig3g-rem-total | reproduced |
| fig3g-rem-bouts | reproduced |
| fig3g-rem-bout-duration | partially reproduced |
| fig3h-nrem-total | reproduced |
| fig3h-nrem-bouts | partially reproduced |
| fig3i-wake-total | partially reproduced |
| fig3i-wake-bouts | partially reproduced |
| fig4a-rem-spectrum | reproduced |
| fig4b-nrem-beta | not reproduced |
| fig4c-wake-power | not reproduced |
| fig5a-swd-state-distribution | partially reproduced |
| fig5a-het-more-nrem-swds | blocked (data missing: one WT with SWDs) |
| fig5c-transition-swds | partially reproduced |
| figs5b-transition-percent | reproduced |
| fig5f-swd-wake-correlation | reproduced |
| fig5def-no-other-correlations | partially reproduced |
| fig1c-glun2b-reduced, fig1df-other-proteins-unchanged | blocked (data missing) |
| figs4-normal-development, figs1-scorer-agreement | blocked (data missing) |
| fig6b, fig6c, fig6d, fig7c (pharmacology) | blocked (data missing) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

**How the analysis differs from the paper's.**
- Two of the paper's nine WT rats are in the dandiset, so every genotype test has two WT, and one
  of them has no SWDs. Several outcomes are partial because the direction holds without p < 0.05.
- The dandiset does not say which rats the paper analysed; all 16 with a genotype are used, and
  GRIN2B-132 (no genotype) is added to each group as a variant. Without GRIN2B-228 (Het, 1 SWD per
  day) fig5c-transition-swds and fig5def-no-other-correlations would meet their criteria, and
  fig3d-rem-hourly and fig3g-rem-total would not (p = 0.060). Without GRIN2B-238 one WT remains,
  and the genotype by hour interaction of fig3d-rem-hourly has p = 0.0001.
- The criterion of figs5b-transition-percent first took the state of an SWD from the epoch before
  it; it was corrected after the first verification round to the Supplementary Methods, which
  judge initiation in NREM on the six epochs before. The first rule gives 36.3%, outside tolerance.
- SWD epochs are the epochs SWD events overlap (the Methods); the lab's scorer uses its per-epoch
  flags, which cover about half of these epochs. The wake bout results depend on this choice: with
  bouts not split by SWD epochs, or with the stored scores, Het have fewer and longer wake bouts,
  and the correlations of SWD time with wake bouts disappear.
- Spectra are computed from about 4.8 h of raw EEG per rat (a median of 5 to 10% of each state's
  epochs) on the SWD-detection channel, with band limits and a Welch segment length chosen here.
  The spectral-slope rejection, as read here, removes no epoch; the per-animal cleaner S1 channel
  is a variant.
- Mixed models are tested as split-plot ANOVAs, which give the same genotype and interaction F as
  a fitted random-intercept model; the sex ANOVA has no interaction term.
- Bouts, the state before an SWD and wake-NREM transitions follow definitions chosen here.

**What was found about the dandiset.**
- Genotype and sex are missing for GRIN2B-132. The dandiset holds 2 of the paper's 9 WT and 14 or 15
  of its 15 Het, and does not say which rats the paper analysed.
- The `sleep_states` description promises code 4 for SWD epochs, which no file holds.
- `swd_epoch_counts` holds flags (4 and 6), not counts; the flags cover about half of the epochs
  SWD events overlap, mostly the last epoch of each SWD.
- The `seizure_totals_by_zt` columns are shifted: per-hour counts are in `spike_wave_discharge`,
  and the day's total is in the first row of `number_of_seizure_events` and `Day`.
- The stored spectra are the scorer's log amplitude referenced to the 5th percentile, not the
  baseline-corrected PSD of their description.
- The raw EEG is stored as unsigned 12-bit codes with an offset of 0 (about +8 mV of DC), and signal
  loss is written as 32767 on all channels, which the description does not mention; up to 73% of
  epochs in the windows read are affected.
- The pharmacology, western blot, development and visual-scoring data are not in the dandiset, and
  there is no published version.